# RENAMING (adding prefixes to maintain data integirty) & NORMALIZING DATA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
print("Installing bcftools...")
!sudo apt-get update -q
!sudo apt-get install bcftools -y -q

Installing bcftools...
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,293 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,887 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Pac

In [ ]:
!pip install cyvcf2 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.8 MB/s eta 0:00:00


In [ ]:
import os
import subprocess
from cyvcf2 import VCF

In [ ]:
# ====================  FILE PATHS ====================
# Directories
INPUT_DIR = '/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered'
OUTPUT_DIR = '/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered_normalized'
REFERENCE_FASTA = '/content/drive/MyDrive/FYP_DATA/DATA/raw_data/reference_fasta/hg38_dna.fa'

# Test with chr1 first
chr_num = "1"

# Input: Filtered Indian samples VCF
input_vcf = os.path.join(INPUT_DIR, f'chr{chr_num}_sas_filtered.vcf.gz')

# Output: Normalized VCF
output_vcf = os.path.join(OUTPUT_DIR, f'chr{chr_num}_sas_normalized.vcf.gz')

# Temporary files
temp_chr_renamed = os.path.join(OUTPUT_DIR, f'temp_chr{chr_num}_renamed.vcf.gz')
# =========================================================

print("=" * 80)
print(f"SG10K NORMALIZATION PIPELINE - CHR{chr_num}")
print("=" * 80)
print(f"Input:  {input_vcf}")
print(f"Output: {output_vcf}")
print(f"Reference: {REFERENCE_FASTA}")

SG10K NORMALIZATION PIPELINE - CHR1
Input:  /content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered/chr1_sas_filtered.vcf.gz
Output: /content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered_normalized/chr1_sas_normalized.vcf.gz
Reference: /content/drive/MyDrive/FYP_DATA/DATA/raw_data/reference_fasta/hg38_dna.fa


In [ ]:
# ============================================================
# STEP 1: Add "chr" prefix to chromosome names
# ============================================================
print("\n" + "=" * 80)
print("STEP 1/4: Adding 'chr' prefix to chromosome names...")
print("=" * 80)

# Create chromosome rename file (only chr1-22 for SG10K)
chr_rename_file = '/tmp/chr_rename_sg10k.txt'
with open(chr_rename_file, 'w') as f:
    for i in range(1, 23):
        f.write(f"{i} chr{i}\n")

print(f"📝 Renaming: {chr_num} → chr{chr_num}")

rename_cmd = f"""
bcftools annotate \
    --rename-chrs {chr_rename_file} \
    -O z \
    -o {temp_chr_renamed} \
    {input_vcf}
"""

print("⏱️  Renaming chromosomes...")
result = subprocess.run(
    rename_cmd,
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("\n❌ CHROMOSOME RENAMING FAILED!")
    print(result.stderr)
    raise Exception("Chromosome renaming failed")

print("✓ Chromosomes renamed")


STEP 1/4: Adding 'chr' prefix to chromosome names...
📝 Renaming: 1 → chr1
⏱️  Renaming chromosomes...
✓ Chromosomes renamed


In [ ]:
# ============================================================
# STEP 2: Index renamed VCF
# ============================================================
print("\n" + "=" * 80)
print("STEP 2/4: Indexing renamed VCF...")
print("=" * 80)

print("   Creating index for renamed file...")
index_result = subprocess.run(
    f"bcftools index -t -f {temp_chr_renamed}",
    shell=True,
    capture_output=True,
    text=True
)

if index_result.returncode != 0:
    print(f"   ❌ Indexing failed!")
    print(f"   Error: {index_result.stderr}")
    raise Exception("Failed to create index")
else:
    print(f"   ✓ Index created")

# Verify index exists
if os.path.exists(f"{temp_chr_renamed}.tbi"):
    index_size = os.path.getsize(f"{temp_chr_renamed}.tbi") / 1024
    print(f"   ✓ Index file verified ({index_size:.1f} KB)")
else:
    print("   ❌ Index file not found!")
    raise Exception("Failed to create index")


STEP 2/4: Indexing renamed VCF...
   Creating index for renamed file...
   ✓ Index created
   ✓ Index file verified (194.6 KB)


In [ ]:
# ============================================================
# STEP 3: Normalize VCF
# ============================================================
print("\n" + "=" * 80)
print("STEP 3/4: Normalizing VCF...")
print("=" * 80)
print("📋 Operations:")
print("   - Left-align indels")
print("   - Split multi-allelic variants (-m-)")
print("   - Check reference alleles")

norm_cmd = f"""
bcftools norm \
    -m- \
    -f {REFERENCE_FASTA} \
    --check-ref w \
    -O z \
    -o {output_vcf} \
    {temp_chr_renamed}
"""

print("\n⏱️  Normalizing (this may take several minutes)...")
result = subprocess.run(
    norm_cmd,
    shell=True,
    capture_output=True,
    text=True
)


STEP 3/4: Normalizing VCF...
📋 Operations:
   - Left-align indels
   - Split multi-allelic variants (-m-)
   - Check reference alleles

⏱️  Normalizing (this may take several minutes)...


In [ ]:
# ============================================================
# STEP 4: Index normalized VCF
# ============================================================
print("\n" + "=" * 80)
print("STEP 4/4: Indexing normalized VCF...")
print("=" * 80)

index_cmd = f"bcftools index -t -f {output_vcf}"

result = subprocess.run(
    index_cmd,
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(f"✓ Index created: {output_vcf}.tbi")
else:
    print("❌ Indexing failed!")
    print(result.stderr)
    raise Exception("Indexing failed")


STEP 4/4: Indexing normalized VCF...
✓ Index created: /content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered_normalized/chr1_sas_normalized.vcf.gz.tbi


In [ ]:
#============================================================
# STEP 5: Verify normalization
# ============================================================
print("\n" + "=" * 80)
print("VERIFICATION: Checking normalized VCF...")
print("=" * 80)

vcf_norm = VCF(output_vcf)

# Check variants
chromosomes_found = set()
multi_count = 0
total = 0

print("\n⏱️  Scanning variants (checking first 100,000)...")

for i, variant in enumerate(vcf_norm):
    chromosomes_found.add(variant.CHROM)
    if len(variant.ALT) > 1:
        multi_count += 1
    total += 1

    # Only check first 100k for speed
    if i >= 100000:
        print(f"   (Checked first {total:,} variants for verification)")
        break

print(f"\n📊 Verification Results:")
print(f"   Chromosomes found: {sorted(chromosomes_found)}")
print(f"   Multi-allelic variants: {multi_count:,} / {total:,}")

if multi_count == 0:
    print("   ✅ All checked variants are properly split!")
elif multi_count < total * 0.01:  # Less than 1%
    print(f"   ⚠️  Found {multi_count} multi-allelic variants ({(multi_count/total)*100:.2f}%)")
else:
    print(f"   ❌ High number of multi-allelic variants: {multi_count}")

# File size comparison
input_size = os.path.getsize(input_vcf) / (1024 * 1024)
output_size = os.path.getsize(output_vcf) / (1024 * 1024)

print(f"\n📁 File Sizes:")
print(f"   Input (filtered):    {input_size:.2f} MB")
print(f"   Output (normalized): {output_size:.2f} MB")
print(f"   Difference: {output_size - input_size:+.2f} MB")



VERIFICATION: Checking normalized VCF...

⏱️  Scanning variants (checking first 100,000)...
   (Checked first 100,001 variants for verification)

📊 Verification Results:
   Chromosomes found: ['chr1']
   Multi-allelic variants: 0 / 100,001
   ✅ All checked variants are properly split!

📁 File Sizes:
   Input (filtered):    522.98 MB
   Output (normalized): 527.24 MB
   Difference: +4.26 MB


In [ ]:
# ============================================================
# STEP 6: Cleanup temporary files
# ============================================================
print("\n" + "=" * 80)
print("CLEANUP: Removing temporary files...")
print("=" * 80)

try:
    if os.path.exists(temp_chr_renamed):
        os.remove(temp_chr_renamed)
        print(f"✓ Removed: {temp_chr_renamed}")
    if os.path.exists(f"{temp_chr_renamed}.tbi"):
        os.remove(f"{temp_chr_renamed}.tbi")
        print(f"✓ Removed: {temp_chr_renamed}.tbi")
except Exception as e:
    print(f"⚠️  Cleanup warning: {e}")

print("\n" + "=" * 80)
print("🎉 NORMALIZATION COMPLETE!")
print("=" * 80)
print(f"Final output: {output_vcf}")
print(f"Index file: {output_vcf}.tbi")
print("=" * 80)


CLEANUP: Removing temporary files...
✓ Removed: /content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered_normalized/temp_chr1_renamed.vcf.gz
✓ Removed: /content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered_normalized/temp_chr1_renamed.vcf.gz.tbi

🎉 NORMALIZATION COMPLETE!
Final output: /content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered_normalized/chr1_sas_normalized.vcf.gz
Index file: /content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered_normalized/chr1_sas_normalized.vcf.gz.tbi


# IGNORE THE CELLS BELOW

In [ ]:
# Quick inspection of normalized chr1
normalized_vcf = '/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered_normalized/chr1_sas_normalized.vcf.gz'

print("=" * 80)
print("INSPECTING NORMALIZED CHR1")
print("=" * 80)

# Check first 10 variants
!bcftools view -H "{normalized_vcf}" | head -10

print("\nCheck chromosome names:")
!bcftools query -f '%CHROM\n' "{normalized_vcf}" | uniq

print("\nCheck if AF, AC, AN are still present:")
!bcftools query -f '%CHROM:%POS\tAC=%AC\tAN=%AN\tAF=%AF\n' "{normalized_vcf}" | head -5

INSPECTING NORMALIZED CHR1
chr1	10333	1_10333_CT_C	CT	C	.	PASS	AR2=0.67;DR2=0.72;AF=0;AC=0;AN=2250	GT	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|

In [ ]:
##########################

In [ ]:
# Quick inspection of normalized chr1
normalized_vcf = '/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered/chr1_sas_filtered.vcf.gz'

print("=" * 80)
print("INSPECTING NORMALIZED CHR1")
print("=" * 80)

# Check first 10 variants
!bcftools view -H "{normalized_vcf}" | head -10

print("\nCheck chromosome names:")
!bcftools query -f '%CHROM\n' "{normalized_vcf}" | uniq

print("\nCheck if AF, AC, AN are still present:")
!bcftools query -f '%CHROM:%POS\tAC=%AC\tAN=%AN\tAF=%AF\n' "{normalized_vcf}" | head -5

INSPECTING NORMALIZED CHR1
1	10333	1_10333_CT_C	CT	C	.	PASS	AR2=0.67;DR2=0.72;AF=0;AC=0;AN=2250	GT	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0

In [1]:
!git clone https://github.com/MusW02/MissenseImpact-FYP.git

Cloning into 'MissenseImpact-FYP'...
fatal: could not read Username for 'https://github.com': No such device or address
